# Objectif retrouver bon bras de levier apres translations CC

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [4]:
# =========================
# INPUTS
# =========================

sbet_path = "/media/b085164/Elements/CALIB_26_02_25/ODyN_calib/base_v0/in/reference.out"

# Intervalles donnés
t1_min, t1_max = 305087.0, 305089.0
t2_min, t2_max = 305982.0, 305985.0

# Translations ICP mesurées (mapping frame)
t_icp_1_map = np.array([-0.050, -0.021, 0.023], dtype=float)
t_icp_2_map = np.array([+0.043, +0.018, 0.023], dtype=float)

# Lever arm actuel (A REMPLIR)
lever_arm_old = np.array([
    0.251,   # x body
    0.008,   # y body
   -0.719,   # z body
], dtype=float)
mapping_frame = "ENU"

In [10]:
from lib.loaders import *
import lib.rotations as rot
cfg_sbet = {
    "type": "SBET"
    # "timeRef": "UTC"   # décommente seulement si ton .out est en UTC
}

time_sbet, lla_sbet, ecef_sbet, rpy_sbet = loadSBET(sbet_path, cfg_sbet)

print("Loaded SBET:")
print("  n samples :", len(time_sbet))
print("  time min  :", time_sbet.min())
print("  time max  :", time_sbet.max())
print("  rpy shape :", rpy_sbet.shape)

Loading file /media/b085164/Elements/CALIB_26_02_25/ODyN_calib/base_v0/in/reference.out
Loaded SBET:
  n samples : 556368
  time min  : 304508.0
  time max  : 307276.0
  rpy shape : (556368, 3)


In [11]:
def mean_angle_rad(a):
    a = np.asarray(a, dtype=float)
    return np.arctan2(np.mean(np.sin(a)), np.mean(np.cos(a)))

def extract_time_window(time, rpy, tmin, tmax):
    mask = (time >= tmin) & (time <= tmax)
    if not np.any(mask):
        raise RuntimeError(f"Aucun échantillon SBET dans [{tmin}, {tmax}]")
    return time[mask], rpy[mask]

def mean_attitude_from_rpy(rpy_window):
    roll_mean = mean_angle_rad(rpy_window[:, 0])
    pitch_mean = mean_angle_rad(rpy_window[:, 1])
    yaw_mean = mean_angle_rad(rpy_window[:, 2])
    return np.array([roll_mean, pitch_mean, yaw_mean], dtype=float)

def rpy_to_R_map_from_body(rpy_mean, mapping_frame="ENU"):
    r, p, y = rpy_mean

    # D'après ton script:
    # R1R2R3transp(r,p,y) donne R_b2ned
    R_b2ned = rot.R1R2R3transp(r, p, y)

    if mapping_frame.upper() == "NED":
        return R_b2ned

    elif mapping_frame.upper() == "ENU":
        # T() convertit ENU <-> NED
        # donc vect_enu = T @ vect_ned
        return rot.T() @ R_b2ned

    else:
        raise ValueError("mapping_frame must be 'ENU' or 'NED'")

def map_translation_to_body(delta_t_map, R_map_from_body):
    return R_map_from_body.T @ delta_t_map

In [12]:
time_1, rpy_1 = extract_time_window(time_sbet, rpy_sbet, t1_min, t1_max)
time_2, rpy_2 = extract_time_window(time_sbet, rpy_sbet, t2_min, t2_max)

rpy_mean_1 = mean_attitude_from_rpy(rpy_1)
rpy_mean_2 = mean_attitude_from_rpy(rpy_2)

R_map_from_body_1 = rpy_to_R_map_from_body(rpy_mean_1, mapping_frame=mapping_frame)
R_map_from_body_2 = rpy_to_R_map_from_body(rpy_mean_2, mapping_frame=mapping_frame)

print("Passage 1 mean attitude:")
print(f"  roll  = {rpy_mean_1[0]:+.8f} rad  ({np.degrees(rpy_mean_1[0]):+.4f} deg)")
print(f"  pitch = {rpy_mean_1[1]:+.8f} rad  ({np.degrees(rpy_mean_1[1]):+.4f} deg)")
print(f"  yaw   = {rpy_mean_1[2]:+.8f} rad  ({np.degrees(rpy_mean_1[2]):+.4f} deg)")

print("\nPassage 2 mean attitude:")
print(f"  roll  = {rpy_mean_2[0]:+.8f} rad  ({np.degrees(rpy_mean_2[0]):+.4f} deg)")
print(f"  pitch = {rpy_mean_2[1]:+.8f} rad  ({np.degrees(rpy_mean_2[1]):+.4f} deg)")
print(f"  yaw   = {rpy_mean_2[2]:+.8f} rad  ({np.degrees(rpy_mean_2[2]):+.4f} deg)")

Passage 1 mean attitude:
  roll  = +0.02642852 rad  (+1.5142 deg)
  pitch = +0.05617019 rad  (+3.2183 deg)
  yaw   = +0.69399853 rad  (+39.7632 deg)

Passage 2 mean attitude:
  roll  = -0.04146764 rad  (-2.3759 deg)
  pitch = -0.01794296 rad  (-1.0281 deg)
  yaw   = -2.43851350 rad  (-139.7165 deg)


In [14]:
dl_body_1 = map_translation_to_body(t_icp_1_map, R_map_from_body_1)
dl_body_2 = map_translation_to_body(t_icp_2_map, R_map_from_body_2)

dl_body_mean = 0.5 * (dl_body_1 + dl_body_2)
dl_body_halfdiff = 0.5 * (dl_body_2 - dl_body_1)

print("Correction estimée depuis passage 1 (body frame):")
print(dl_body_1)

print("\nCorrection estimée depuis passage 2 (body frame):")
print(dl_body_2)

print("\nMoyenne des deux (body frame):")
print(dl_body_mean)

print("\nDemi-différence (cohérence):")
print(dl_body_halfdiff)

Correction estimée depuis passage 1 (body frame):
[-0.04675625 -0.02567229 -0.02499573]

Correction estimée depuis passage 2 (body frame):
[-0.04193987 -0.02022388 -0.02310936]

Moyenne des deux (body frame):
[-0.04434806 -0.02294808 -0.02405255]

Demi-différence (cohérence):
[0.00240819 0.0027242  0.00094319]


In [15]:
lever_arm_new = lever_arm_old + dl_body_mean

print("======================================")
print("LEVER ARM UPDATE")
print("======================================")
print(f"old lever arm : {lever_arm_old}")
print(f"delta mean    : {dl_body_mean}")
print(f"new lever arm : {lever_arm_new}")

print("\nYAML:")
print("lever_arm:")
print(f"  x: {lever_arm_new[0]:.6f}")
print(f"  y: {lever_arm_new[1]:.6f}")
print(f"  z: {lever_arm_new[2]:.6f}")

LEVER ARM UPDATE
old lever arm : [ 0.251  0.008 -0.719]
delta mean    : [-0.04434806 -0.02294808 -0.02405255]
new lever arm : [ 0.20665194 -0.01494808 -0.74305255]

YAML:
lever_arm:
  x: 0.206652
  y: -0.014948
  z: -0.743053


In [16]:
t_mean_map = 0.5 * (t_icp_1_map + t_icp_2_map)
t_halfdiff_map = 0.5 * (t_icp_2_map - t_icp_1_map)

print("======================================")
print("DIAGNOSTIC")
print("======================================")
print(f"mapping_frame     = {mapping_frame}")
print(f"t_icp_1_map       = {t_icp_1_map}")
print(f"t_icp_2_map       = {t_icp_2_map}")
print(f"mean map          = {t_mean_map}")
print(f"half-diff map     = {t_halfdiff_map}")
print(f"||dl_body_1||     = {np.linalg.norm(dl_body_1):.4f} m")
print(f"||dl_body_2||     = {np.linalg.norm(dl_body_2):.4f} m")
print(f"||dl_body_mean||  = {np.linalg.norm(dl_body_mean):.4f} m")
print(f"||dl_body_hdiff|| = {np.linalg.norm(dl_body_halfdiff):.4f} m")

DIAGNOSTIC
mapping_frame     = ENU
t_icp_1_map       = [-0.05  -0.021  0.023]
t_icp_2_map       = [0.043 0.018 0.023]
mean map          = [-0.0035 -0.0015  0.023 ]
half-diff map     = [0.0465 0.0195 0.    ]
||dl_body_1||     = 0.0589 m
||dl_body_2||     = 0.0520 m
||dl_body_mean||  = 0.0554 m
||dl_body_hdiff|| = 0.0038 m
